## 0. Task

Multi-label identification of **100 masked** statute IDs. Macro-F1. Mentions of the statutes are stripped from the text, so the model cannot copy section numbers.


In [ ]:
import os
import sys
from pathlib import Path

HERE = Path.cwd().resolve()
SEARCH = [HERE, HERE.parent, Path("/kaggle/working")]
for base in (HERE, HERE.parent, HERE.parent.parent, HERE.parent.parent.parent):
    SEARCH.append(base)
    for phase in range(1, 8):
        SEARCH.append(base / f"phase{phase}")
        SEARCH.append(base / f"phase{phase}" / "code")
        SEARCH.append(base / "code" / f"phase{phase}")
for folder in SEARCH:
    if folder.is_dir() and str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
print("python path heads:")
for folder in sys.path[:8]:
    print(" ", folder)


## 1. Neural head

Up to six 256-token windows. Per-label attention over tokens, plus a mean-pooled linear path whose bias is the label-frequency prior. BCE loss. Empty predictions are replaced by the label with the largest score/threshold ratio.


In [ ]:
print("Windows: 6 x 256 from the efficient body (LoRMA and other PEFT).")
print("One query vector per statute ID — not a 100-way softmax.")


## 2. Per-label thresholds

A global cut of 0.5 cannot serve both the head and the tail of the frequency histogram. Thresholds are fit **per label** on a two-fold split of dev.


In [ ]:
print("See phase5_lsi.py: per_label_cuts, apply_cuts")
print("Do not quote train+dev 0.57 — that fit sees the labels it reports.")


## 3. TF-IDF blend (selected)

A TF-IDF + per-label logistic regression on the same corpus beats a 256-token neural view of a long judgment. The selected zip **adds** the neural logits to the linear model rather than replacing it.


In [ ]:
print("scripts : phase5_tfidf.py  +  phase5_fineblend.py / phase5_final_blend.py")
print("idea    : standardize both logit matrices, weighted sum, then per-label τ")
print("selected serving path is the blend, not PEFT-only")


## 4. Zip


In [ ]:
print("prediction.json: list of {id, pred: [statute_id, ...]}")
